In [13]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [94]:
data = pd.read_csv("mnist_test.csv")
data.shape 
X = data.drop(columns = ["label"])
y = data['label']
X_train, X_test, y_train, y_test = train_test_split(np.array(X),np.array(y), test_size = 0.1, shuffle=True)
len(X_train), len(X_test)

(9000, 1000)

In [ ]:
def relu(X):
    return np.maximum(0, X)
def softmax(X):
    expX = np.exp(X - np.max(X, axis=0, keepdims=True))  
    return expX / np.sum(expX, axis=0, keepdims=True)

def relu_der(X):
    return (X > 0).astype(float)

def initialization():
    W1 = np.random.randn(10, 784)  * np.sqrt(1/784)
    B1 = np.random.randn(10,1)
    W2 = np.random.randn(10, 10) * np.sqrt(1/10)
    B2 = np.random.randn(10,1)
    return W1, B1, W2, B2


def forward(X, W1, B1, W2, B2):
    Z1 = W1.dot(X) + B1
    A1 = relu(Z1)
    Z2 = W2.dot(A1)+ B2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2


def update_weights(W1, B1, W2, B2, dw1, db1, dw2, db2, a):
    W1 -= dw1 * a
    B1 -= db1 * a
    W2 -= dw2 * a
    B2 -= db2 * a
    return W1, B1, W2, B2

def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, Y.max()+1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    return one_hot_Y.T


def backward(Z1, A1, A2, W2, X, Y):
    m = Y.size
    one_hot_y = one_hot(Y)
    DZ2 = A2 - one_hot_y
    dw2 = (1/m) * DZ2.dot(A1.T)
    db2 = (1/m) * np.sum(DZ2, 1, keepdims=True)
    DZ1 = W2.T.dot(DZ2) * relu_der(Z1)
    dw1 = (1/m) * DZ1.dot(X.T)
    db1 = (1/m) * np.sum(DZ1, 1, keepdims=True)

    return dw1, db1, dw2, db2
    
def get_prediction(A2):
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
    return np.sum(predictions==Y)/ Y.size

def gradient_descent(X, Y, a=0.1, iterations=100):
    X = X.T / 255.0
    W1, B1, W2, B2 = initialization()
    for i in range(iterations):
        Z1, A1, _, A2 = forward(X, W1, B1, W2, B2)
        dw1, db1, dw2, db2 = backward(Z1, A1, A2, W2, X, Y)
        W1, B1, W2, B2 = update_weights(W1, B1, W2, B2, dw1, db1, dw2, db2, a)
        if (i+1) % 25 == 0:
            print(f"Epoch {i+1}")
            print(f'Accuracy: {get_accuracy(get_prediction(A2),Y)}')
    return W1, B1, W2, B2


In [109]:
W1, B1, W2, B2 = gradient_descent(X_train, y_train, iterations=300)

Epoch 25
Accuracy: 0.5446666666666666
Epoch 50
Accuracy: 0.7261111111111112
Epoch 75
Accuracy: 0.7965555555555556
Epoch 100
Accuracy: 0.8323333333333334
Epoch 125
Accuracy: 0.8475555555555555
Epoch 150
Accuracy: 0.859
Epoch 175
Accuracy: 0.8665555555555555
Epoch 200
Accuracy: 0.872
Epoch 225
Accuracy: 0.877
Epoch 250
Accuracy: 0.8822222222222222
Epoch 275
Accuracy: 0.8865555555555555
Epoch 300
Accuracy: 0.8897777777777778


In [137]:
def predict(X, W1=W1, B1=B1, W2=W2, B2=B2):
    X = X.reshape(-1, 1) / 255.0
    _,_,_, A2 = forward(X, W1, B1, W2, B2)
    predict= get_prediction(A2)
    return predict

In [124]:
get_accuracy(predict(X_test),y_test)

np.float64(0.899)

In [150]:
i = np.random.randint(1, 1000)

predict(X_test[i]), y_test[i]

(array([6]), np.int64(6))